In [ ]:
# --------------------------
# 1. 导入GUI和图像处理依赖库
# --------------------------
import tkinter as tk  # Python自带GUI库，用于创建窗口和画布
import tkinter.font as tk_font
from tkinter import Label, Button  # Label：显示文字；Button：按钮（清空/预测）
from PIL import Image, ImageDraw  # PIL：图像处理库（Image：图像操作；ImageDraw：绘制图像）
import numpy as np  # 数组操作（将图像转成模型可输入的格式）
from keras.models import load_model  # 加载之前保存的MNIST模型


# --------------------------
# 2. 定义手写识别GUI类（封装窗口、画布、功能）
# --------------------------
class MNISTHandwritingGUI:
    def __init__(self, root):
        """
        初始化GUI窗口和组件
        root：Tkinter的主窗口对象
        """
        # ------------ 基础配置 ------------
        self.root = root
        self.root.title("MNIST Handwriting Recognition")  # 窗口标题（英文）
        self.root.geometry("500x600")  # 窗口大小（宽x高）
        self.root.resizable(False, False)  # 禁止窗口缩放（避免画布变形）

        # ------------ 画布配置（手写区域） ------------
        self.canvas_width = 400  # 画布宽度
        self.canvas_height = 400  # 画布高度
        # 创建画布：背景色为白色（模拟纸），边框为黑色
        self.canvas = tk.Canvas(
            root, 
            width=self.canvas_width, 
            height=self.canvas_height, 
            bg="white", 
            bd=2, 
            relief="solid"
        )
        self.canvas.pack(pady=20)  # 画布在窗口中居中，上下留20像素间距

        # ------------ 图像缓存（用于存储手写内容） ------------
        # 创建一个与画布同尺寸的PIL图像（模式为L：灰度图像，单通道）
        self.image = Image.new("L", (self.canvas_width, self.canvas_height), 255)
        # 创建ImageDraw对象，用于在PIL图像上绘制（模拟手写笔画）
        self.draw = ImageDraw.Draw(self.image)

        # ------------ 绑定鼠标事件（实现手写功能） ------------
        # 鼠标按下：记录起始位置，开始绘制
        self.canvas.bind("<Button-1>", self.start_drawing)
        # 鼠标拖动：随鼠标移动绘制线条
        self.canvas.bind("<B1-Motion>", self.draw_line)
        # 鼠标释放：结束绘制（可选，此处仅记录状态）
        self.canvas.bind("<ButtonRelease-1>", self.end_drawing)

        # ------------ 状态变量（记录鼠标位置） ------------
        self.last_x = None  # 上一次鼠标的x坐标
        self.last_y = None  # 上一次鼠标的y坐标

        # ------------ 结果显示区域 ------------
        # 标签：显示“预测结果”提示文字（英文）
        self.result_label = Label(
            root, 
            text="Prediction: Unrecognized",  # 初始文本（英文）
            font=("Arial", 16),  # 使用Arial字体（英文无乱码）
            fg="blue"
        )
        self.result_label.pack(pady=10)  # 标签在画布下方

        # ------------ 功能按钮 ------------
        # 按钮1：清空画布（点击后清除手写内容，文字为英文）
        self.clear_btn = Button(
            root, 
            text="Clear Canvas",  # 按钮文字（英文）
            font=("Arial", 12),  # 使用Arial字体
            command=self.clear_canvas,
            width=15,
            height=2
        )
        self.clear_btn.pack(side=tk.LEFT, padx=50, pady=10)  # 按钮居左

        # 按钮2：预测数字（点击后识别手写内容，文字为英文）
        self.predict_btn = Button(
            root, 
            text="Recognize Digit",  # 按钮文字（英文）
            font=("Arial", 12),  # 使用Arial字体
            command=self.predict_digit,
            width=15,
            height=2
        )
        self.predict_btn.pack(side=tk.RIGHT, padx=50, pady=10)  # 按钮居右

        # ------------ 加载训练好的模型 ------------
        try:
            # 加载第一步保存的模型（若模型文件不在当前目录，需写完整路径）
            self.model = load_model("muist训练模型.h5")
            print("模型加载成功！")  # 控制台输出保留中文（不影响GUI）
        except Exception as e:
            # 若模型加载失败（如文件不存在），弹出提示并关闭窗口（提示文字为英文）
            Label(root, text=f"Model loading failed: {str(e)}", font=("Arial", 12), fg="red").pack()
            self.root.after(3000, self.root.destroy)  # 3秒后自动关闭窗口


    # --------------------------
    # 3. 鼠标事件函数（实现手写绘制）
    # --------------------------
    def start_drawing(self, event):
        """鼠标按下时触发：记录起始位置"""
        # event.x, event.y：鼠标在画布上的坐标
        self.last_x = event.x
        self.last_y = event.y

    def draw_line(self, event):
        if self.last_x is not None and self.last_y is not None:
            # 1. 在Tkinter画布上绘制可见线条（用户看到的手写效果）
            self.canvas.create_line(
                self.last_x, self.last_y,
                event.x, event.y,
                fill="black",
                width=8,
                capstyle=tk.ROUND,  # 线条端点圆润，避免锯齿
                smooth=tk.TRUE  # 线条平滑
            )
    
            # 2. 调整PIL椭圆的坐标（确保x0 ≤ x1，y0 ≤ y1）
            x0 = min(self.last_x - 4, event.x + 4)
            y0 = min(self.last_y - 4, event.y + 4)
            x1 = max(self.last_x - 4, event.x + 4)
            y1 = max(self.last_y - 4, event.y + 4)
    
            # 绘制椭圆（现在坐标范围合法）
            self.draw.ellipse(
                [x0, y0, x1, y1],
                fill=0  # 填充黑色（对应MNIST的数字颜色）
            )
    
            # 更新最后一次鼠标位置
            self.last_x = event.x
            self.last_y = event.y

    def end_drawing(self, event):
        """鼠标释放时触发：重置位置记录"""
        self.last_x = None
        self.last_y = None


    # --------------------------
    # 4. 图像预处理函数（将手写图像转成模型可识别的格式）
    # --------------------------
    def preprocess_image(self):
        """
        预处理步骤：
        1. 缩小图像到28x28（匹配MNIST数据集的尺寸）
        2. 反色（确保数字为黑色、背景为白色，与MNIST一致）
        3. 归一化（像素值从0-255缩放到0-1）
        4. 展平成1D向量（784维，匹配模型输入）
        """
        # 1. 缩小图像到28x28（抗锯齿，保持图像清晰）
        resized_img = self.image.resize((28, 28), Image.Resampling.LANCZOS)

        # 2. 转成numpy数组（便于数值操作）
        img_array = np.array(resized_img, dtype=np.float32)

        # 3. 反色（MNIST是“黑底白字”，手写是“白底黑字”，需反色）
        img_array = 255 - img_array

        # 4. 归一化（像素值缩放到0-1，与训练时的预处理一致）
        img_array /= 255.0

        # 5. 展平成1D向量（从(28,28)转成(1,784)，1表示批量大小为1）
        img_flattened = img_array.reshape(1, 784)

        return img_flattened


    # --------------------------
    # 5. 预测函数（调用模型识别手写数字）
    # --------------------------
    def predict_digit(self):
        """识别手写内容，更新结果标签"""
        try:
            # 1. 预处理手写图像
            processed_img = self.preprocess_image()

            # 2. 模型预测（输出10个数字的概率）
            prediction_proba = self.model.predict(processed_img, verbose=0)

            # 3. 取概率最大的索引作为预测结果（索引0-9对应数字0-9）
            predicted_digit = np.argmax(prediction_proba)

            # 4. 计算预测概率（显示置信度，让用户知道识别的可靠性）
            confidence = prediction_proba[0][predicted_digit] * 100  # 转成百分比

            # 5. 更新结果标签（显示预测数字和置信度，文字为英文）
            self.result_label.config(
                text=f"Prediction: {predicted_digit} (Confidence: {confidence:.1f}%)",
                fg="green"  # 预测成功时文字变绿色
            )

        except Exception as e:
            # 若预测失败（如图像为空），显示错误信息（文字为英文）
            self.result_label.config(
                text=f"Recognition failed: {str(e)}",
                fg="red"  # 错误时文字变红色
            )


    # --------------------------
    # 6. 清空画布函数（清除手写内容）
    # --------------------------
    def clear_canvas(self):
        """清空画布和PIL图像缓存，重置结果标签"""
        # 1. 清空Tkinter画布（删除所有绘制的内容）
        self.canvas.delete("all")

        # 2. 重置PIL图像（重新创建白色背景的灰度图像）
        self.image = Image.new("L", (self.canvas_width, self.canvas_height), 255)
        self.draw = ImageDraw.Draw(self.image)

        # 3. 重置结果标签（恢复初始状态，文字为英文）
        self.result_label.config(
            text="Prediction: Unrecognized",
            fg="blue"
        )


# --------------------------
# 7. 启动GUI窗口（程序入口）
# --------------------------
if __name__ == "__main__":
    # 创建Tkinter主窗口
    root = tk.Tk()
    # 初始化手写识别GUI
    app = MNISTHandwritingGUI(root)
    # 启动窗口循环（保持窗口打开，等待用户操作）
    root.mainloop()

2025-11-07 20:37:06.115883: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-07 20:37:06.115980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-07 20:37:06.146408: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


模型加载成功！
